# LSTM Forecaster Training
Train the time-series forecasting model for predicting future spending. The generated model is automatically saved to `./models_store`.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to python path so we can import app modules
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)


In [ ]:
!pip install matplotlib ipywidgets


In [ ]:
import numpy as np
import pandas as pd
from app.models.forecaster import LSTMForecaster
from app.core.config import config
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
# Configuration
DATA_PATH = '../test_data.csv'  # Path to your transactions CSV
LOOKBACK = config.LSTM_LOOKBACK
EPOCHS = config.LSTM_EPOCHS
print(f"Using dataset: {DATA_PATH}")


In [ ]:
# 1. Load Dataset & Aggregate Daily
def load_dataset(path: str) -> pd.Series:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    col_map = {}
    for col in df.columns:
        if 'date' in col and 'date' not in col_map.values():
            col_map[col] = 'date'
        elif ('amount' in col or 'debit' in col or 'credit' in col) and 'amount' not in col_map.values():
            col_map[col] = 'amount'
    df = df.rename(columns=col_map)
    if 'date' not in df.columns or 'amount' not in df.columns:
        raise ValueError("CSV must contain 'date' and 'amount'")
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    df = df.dropna(subset=['date', 'amount'])
    # Aggregate to daily totals
    daily = df.set_index('date')['amount'].resample('D').sum().fillna(0)
    return daily

daily_series = load_dataset(DATA_PATH)
print(f"Loaded {len(daily_series)} days of data.")

plt.figure(figsize=(10, 4))
plt.plot(daily_series.index, daily_series.values)
plt.title('Historical Daily Spending')
plt.show()


In [ ]:
# 2. Train LSTM Model & Persist to app/models_store
if len(daily_series) <= LOOKBACK:
    print(f"ERROR: Need at least {LOOKBACK + 1} days of data.")
else:
    forecaster = LSTMForecaster(lookback=LOOKBACK, epochs=EPOCHS)
    print("Training model...")
    forecaster.fit(daily_series)
    print(f"Model saved to: {forecaster._model_path}")
    print(f"Scaler saved to: {forecaster._scaler_path}")


In [ ]:
# 3. Generate 30-Day Forecast
if len(daily_series) > LOOKBACK:
    forecast = forecaster.predict(daily_series, horizon=30)
    
    # Plot history + forecast
    plt.figure(figsize=(12, 5))
    hist_days = len(daily_series)
    plt.plot(range(hist_days), daily_series.values, label='Historical')
    plt.plot(range(hist_days, hist_days + 30), forecast, label='Forecast', color='orange')
    plt.axvline(x=hist_days, color='black', linestyle='--')
    plt.title('30-Day Spending Forecast')
    plt.legend()
    plt.show()
